# 1、定义一个系统提示词模板

In [1]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import SystemMessage, HumanMessage

SYSTEM_PROMPT = """
你是一位精通天气预报的专家，且说话总是带着双关语（谐音梗）。你可以使用两个工具：
get_weather_for_location：用于获取特定地点的天气情况。
get_user_location：用于获取用户当前所在的地点。
如果用户询问天气，请务必确认地点。如果你能从问题中判断出他们指的是自己所在的任何地方，请使用 get_user_location 工具来获取他们的位置。
"""

# 2、创建工具

In [2]:
# from dataclasses import dataclass
# from langchain.tools import tool, ToolRuntime
#
#
# @tool
# def get_weather_for_location(city: str):
#     """获取给定城市的天气情况"""
#     return f"{city}的天气总是阳光灿烂！"
#
#
# @dataclass
# class Context:
#     """自定义运行时上下文模式"""
#     user_id: str
#
#
# @tool
# def get_user_location(runtime: ToolRuntime[Context]) -> str:
#     """根据用户id检索信息"""
#     user_id = runtime.context.user_id
#     return "北京" if user_id == "1" else "武汉"


import requests
from langchain.tools import tool
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime


@dataclass
class Context:
    """自定义运行时上下文模式"""
    user_id: str


@tool
def get_weather_for_location(city: str) -> str:
    """获取给定城市实时的天气情况。"""
    try:
        # 使用 Nominatim API 获取城市坐标
        geo_url = f"https://nominatim.openstreetmap.org/search?format=json&q={city}&limit=1"
        headers = {'User-Agent': 'WeatherApp/1.0'}
        geo_res = requests.get(geo_url, headers=headers).json()
        
        if not geo_res:
            return f"抱歉，找不到名为 {city} 的城市，它可能躲在'云'深不知处了。"
        
        lat = geo_res[0]['lat']
        lon = geo_res[0]['lon']
        
        # 获取该经纬度的实时天气数据
        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m,relative_humidity_2m,weather_code&hourly=temperature_2m,relative_humidity_2m&timezone=auto"
        weather_res = requests.get(weather_url).json()
        
        temp = weather_res['current']['temperature_2m']
        humidity = weather_res['current']['relative_humidity_2m']
        
        # 获取天气代码对应的描述
        weather_code = weather_res['current']['weather_code']
        weather_desc = {
            0: "晴朗",
            1: "主要晴朗",
            2: "部分多云",
            3: "阴天",
            45: "雾",
            48: "雾凇",
            51: "毛毛雨",
            53: "小雨",
            55: "细雨",
            56: "冻毛毛雨",
            57: "冻细雨",
            61: "雨",
            63: "中雨",
            65: "大雨",
            66: "冻雨",
            67: "冻雨",
            71: "雪",
            73: "小雪",
            75: "大雪",
            77: "雪粒",
            80: "阵雨",
            81: "中阵雨",
            82: "大阵雨",
            85: "小阵雪",
            86: "大阵雪",
            95: "雷暴",
            96: "伴有冰雹的雷暴",
            99: "伴有冰雹的雷暴"
        }.get(weather_code, "未知天气")
        
        return f"{city} 现在的气温是 {temp}℃，湿度为 {humidity}%，天气：{weather_desc}。这天气真是'温'文尔雅！"
        
    except Exception as e:
        return f"哎呀，获取天气的'网'线断了：{str(e)}"


@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """根据用户id检索信息"""
    user_id = runtime.context.user_id
    return "北京" if user_id == "1" else "武汉"

# 3、配置模型

In [3]:
# 方案1：使用指定provider参数
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv

# 获取API密钥
load_dotenv()

ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_BASE_URL = os.getenv("ZHIPUAI_BASE_URL")

model = init_chat_model(
    model="glm-4.5-air",
    model_provider="openai",  # 因为智谱AI兼容OpenAI API
    temperature=0.5,
    timeout=10,
    max_tokens=1000,
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_BASE_URL
)

# 4、定义响应格式

In [4]:
from dataclasses import dataclass


# 我们这里使用了数据类（dataclass），不过也支持使用 Pydantic 模型。
@dataclass
class ResponseFormat:
    """agent的响应格式。"""
    # 包含双关语的回复（必须提供）
    punny_response: str
    # 若有可用的天气相关信息
    weather_conditions: str | None = None

# 5、添加内存

In [5]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

# 6、 创建并运行agent

In [6]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location, get_weather_for_location],
    context_schema=Context,
    response_format=ToolStrategy(ResponseFormat),
    checkpointer=checkpointer
)

# `thread_id` 是特定对话的唯一标识符
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "外面的天气怎么样?"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="Florida is still having a 'sun-derful' day! The sunshine is playing 'ray-dio' hits all day long! I'd say it's the perfect weather for some 'solar-bration'! If you were hoping for rain, I'm afraid that idea is all 'washed up' - the forecast remains 'clear-ly' brilliant!",
#     weather_conditions="It's always sunny in Florida!"
# )


# Note that we can continue the conversation using the same `thread_id`.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "谢谢"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="You're 'thund-erfully' welcome! It's always a 'breeze' to help you stay 'current' with the weather. I'm just 'cloud'-ing around waiting to 'shower' you with more forecasts whenever you need them. Have a 'sun-sational' day in the Florida sunshine!",
#     weather_conditions=None
# )

ResponseFormat(punny_response='北京现在天气晴朗，真是"晴"空万里，心情万里！18.9℃的气温让人感觉"温"暖如春，61%的湿度也刚刚好，简直是"晴"天霹雳般的好天气！', weather_conditions='晴朗')
ResponseFormat(punny_response='不客气！希望这"晴"朗的天气能给你带来"晴"天好心情！如果还有其他需要了解的天气信息，随时告诉我哦～', weather_conditions='晴朗')
